In [2]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm

In [3]:
# Find nearest index
def find_index(array, x):
    if array.ndim == 1:
        idx = np.argmin(np.abs(array - x))
    elif array.ndim == 2:
        idx = np.unravel_index(np.argmin(np.abs(array - x)), array.shape)
    else:
        raise ValueError("Unsupported array dimensions for find_index function.")
    return idx

In [8]:
#Read in Precip (TMQ) timeseries file
nc_daily = xr.open_dataset('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.TMQ.nc', engine='netcdf4')
nc_daily # no conversions required because units of precip are kg/m2/d which is equivalent to mm/day (desired unit)

<xarray.Dataset>
Dimensions:  (time: 7671, lat: 192, lon: 288)
Coordinates:
  * lat      (lat) float64 -90.0 -89.06 -88.12 -87.17 ... 87.17 88.12 89.06 90.0
  * lon      (lon) float64 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
Data variables:
    TMQ      (time, lat, lon) float32 ...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001
    logname:           demurray
    host:              derecho8
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [9]:
#Select cells that correspond to NADP sites: read in NADP lat/long and apply the find nearest function
pathData = '/glade/u/home/demurray/External File Uploads'
os.chdir(pathData)
ntn = pd.read_csv('ntn.csv')

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(ntn))
for index, row in ntn.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_daily['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_daily['lon'].values, lon)
    subset = nc_daily.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_daily_nadp = xr.concat(subset_list, dim='siteId')
nc_daily_nadp


  0%|          | 0/391 [00:09<?, ?it/s]

100%|██████████| 391/391 [00:00<00:00, 470.71it/s]


<xarray.Dataset>
Dimensions:  (siteId: 391, time: 7671)
Coordinates:
    lat      (siteId) float64 57.02 56.07 57.02 65.5 ... 43.82 42.88 39.11 40.99
    lon      (siteId) float64 248.8 248.8 248.8 212.5 ... 271.2 280.0 253.8
  * time     (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
  * siteId   (siteId) <U4 'AB32' 'AB34' 'AB36' 'AK01' ... 'WI99' 'WV99' 'WY96'
Data variables:
    TMQ      (siteId, time) float32 3.055 2.726 5.608 11.05 ... 3.84 4.505 8.97
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001
    logname:           demurray
    host:              derecho8
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [10]:
#Add a loop that for each siteId it writes a file to timeseries
output_directory = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites'

unique_site_ids = nc_daily_nadp['siteId'].values

# Initialize tqdm
pbar = tqdm(unique_site_ids, desc="Writing subset files")

# Iterate over each unique siteId
for site_id in pbar:
    # Subset the dataset for the current siteId
    subset_ds = nc_daily_nadp.where(nc_daily_nadp['siteId'] == site_id, drop=True)
    
    # Construct the filename
    filename = f'{site_id}_DailyTimeseries_TMQ.nc'
    
    # Write the subsetted dataset to the specified directory
    output_path = os.path.join(output_directory, filename)
    subset_ds.to_netcdf(output_path)
    
    # Update tqdm description
    pbar.set_description(f"Writing subset files: {filename}")

Writing subset files: WY96_DailyTimeseries_TMQ.nc: 100%|██████████| 391/391 [00:08<00:00, 44.53it/s]


In [13]:
#Turn nc_daily_nadp xarray into a pandas dataframe with similar attributes to the NADP dataset
mod_nadp = nc_daily_nadp.to_dataframe().reset_index()
mod_nadp = mod_nadp.rename(columns = {'TMQ': 'Mod_Precip_mm'})
mod_nadp.head(10)

,siteId,time,Mod_Precip_mm,lat,lon
0,AB32,2002-01-01,3.054744,57.015707,248.75
1,AB32,2002-01-02,2.726092,57.015707,248.75
2,AB32,2002-01-03,5.607596,57.015707,248.75
3,AB32,2002-01-04,11.050609,57.015707,248.75
4,AB32,2002-01-05,9.330268,57.015707,248.75
5,AB32,2002-01-06,4.834256,57.015707,248.75
6,AB32,2002-01-07,8.845551,57.015707,248.75
7,AB32,2002-01-08,10.161882,57.015707,248.75
8,AB32,2002-01-09,9.931054,57.015707,248.75
9,AB32,2002-01-10,6.976051,57.015707,248.75


In [14]:
#read in timeseries of: NADP NTN and ensure correct/consistent formatting

# subset for NTN comparisons: PRECIP (subppt) (downloaded NADP NTN 2002-2023 for all sites at weekly scale on 3/25/2024 excluding invalids)
pathData = '/glade/u/home/demurray/External File Uploads'
os.chdir(pathData)
nadp_df = pd.read_csv('NTN-ALL-Weekly 2000_2022.csv', parse_dates = ['dateOn', 'dateOff'])
nadp_df = nadp_df[['siteId', 'dateOn', 'dateOff','subppt']]

#replace all negative numbers with np.nan and assign units, perform conversions
nadp_df[nadp_df.select_dtypes(include='number') < 0] = np.nan
nadp_df = nadp_df.rename(columns = {'subppt':'ppt_mm'}) #assign units

#Ensure correct datetime formatting
nadp_df['dateOn'] = pd.to_datetime(nadp_df['dateOn'], format='%m/%d/%Y %H:%M')
nadp_df['dateOff'] = pd.to_datetime(nadp_df['dateOff'], format='%m/%d/%Y %H:%M')
nadp_df = nadp_df.sort_values(['siteId', 'dateOn'], ascending = True)

#Need to round date because using DAILY data for modelled comparisons
nadp_df['dateOnround'] = nadp_df.dateOn + dt.timedelta(hours=12)
nadp_df['dateOnround'] = pd.to_datetime(nadp_df.dateOnround.dt.strftime('%Y-%m-%d'))
nadp_df['dateOffround'] = nadp_df.dateOff + dt.timedelta(hours=12)
nadp_df['dateOffround'] = pd.to_datetime(nadp_df.dateOffround.dt.strftime('%Y-%m-%d'))

#merge with site info and then select relevant columns
nadp_df = pd.merge(nadp_df, ntn, on = 'siteId')
nadp_df = nadp_df[['siteId','latitude', 'longitude', 'dateOn', 'dateOff', 'dateOnround', 'dateOffround', 'ppt_mm']]

nadp_df.head(5)

,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,ppt_mm
0,AB32,57.1894,-111.6406,2016-09-13 18:40:00,2016-09-20 15:10:00,2016-09-14,2016-09-21,0.762
1,AB32,57.1894,-111.6406,2016-09-20 15:15:00,2016-09-28 16:00:00,2016-09-21,2016-09-29,0.508
2,AB32,57.1894,-111.6406,2016-09-28 16:00:00,2016-10-05 16:55:00,2016-09-29,2016-10-06,18.034
3,AB32,57.1894,-111.6406,2016-10-05 16:55:00,2016-10-11 17:00:00,2016-10-06,2016-10-12,10.160
4,AB32,57.1894,-111.6406,2016-10-11 17:00:00,2016-10-18 20:00:00,2016-10-12,2016-10-19,13.208


In [15]:
##Assign sampling intervals to NADP NTN deposition data
nadp_df['SamplingInt'] = pd.Series(dtype='int')
nadp_df['IntTime'] = pd.Series(dtype='int')

sites = nadp_df.siteId.unique()

for i in tqdm(sites, unit = 'sites', total = len(sites), ncols = 100):
    nadp_df.loc[nadp_df.siteId == i,'SamplingInt'] = list(range(0, len(nadp_df.loc[nadp_df.siteId == i]), 1))
    nadp_df.loc[nadp_df.siteId == i, 'IntTime'] = nadp_df.loc[nadp_df.siteId == i, 'dateOff'] - nadp_df.loc[nadp_df.siteId == i, 'dateOn']
nadp_df.head(10)

100%|██████████████████████████████████████████████████████████| 339/339 [00:39<00:00,  8.60sites/s]


,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,ppt_mm,SamplingInt,IntTime
0,AB32,57.1894,-111.6406,2016-09-13 18:40:00,2016-09-20 15:10:00,2016-09-14,2016-09-21,0.762,0.0,6 days 20:30:00
1,AB32,57.1894,-111.6406,2016-09-20 15:15:00,2016-09-28 16:00:00,2016-09-21,2016-09-29,0.508,1.0,8 days 00:45:00
2,AB32,57.1894,-111.6406,2016-09-28 16:00:00,2016-10-05 16:55:00,2016-09-29,2016-10-06,18.034,2.0,7 days 00:55:00
3,AB32,57.1894,-111.6406,2016-10-05 16:55:00,2016-10-11 17:00:00,2016-10-06,2016-10-12,10.160,3.0,6 days 00:05:00
4,AB32,57.1894,-111.6406,2016-10-11 17:00:00,2016-10-18 20:00:00,2016-10-12,2016-10-19,13.208,4.0,7 days 03:00:00
5,AB32,57.1894,-111.6406,2016-10-18 20:00:00,2016-10-25 18:00:00,2016-10-19,2016-10-26,3.556,5.0,6 days 22:00:00
6,AB32,57.1894,-111.6406,2016-10-25 18:00:00,2016-10-31 16:47:00,2016-10-26,2016-11-01,4.572,6.0,5 days 22:47:00
7,AB32,57.1894,-111.6406,2016-10-31 16:47:00,2016-11-07 17:00:00,2016-11-01,2016-11-08,1.524,7.0,7 days 00:13:00
8,AB32,57.1894,-111.6406,2016-11-07 17:00:00,2016-11-15 16:00:00,2016-11-08,2016-11-16,0.254,8.0,7 days 23:00:00
9,AB32,57.1894,-111.6406,2016-11-15 16:00:00,2016-11-22 18:30:00,2016-11-16,2016-11-23,0.508,9.0,7 days 02:30:00


In [ ]:
#Write loop to assign sampling intervals to  modelled data frame
mod_nadp['SamplingInt'] = pd.Series(dtype='int') # Add a SamplingInt column to the modelled
sites =nadp_df.siteId.unique()

for i in tqdm(sites, unit = "sites", total = len(sites), ncols = 100):
    sampleInt = nadp_df.loc[nadp_df.siteId==i,'SamplingInt']
    #print(i)
    for j in sampleInt:
       #print(j)
       begDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOnround'].item())
       endDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOffround'].item())
       #print(endDate)
       mod_nadp.loc[(mod_nadp.siteId == i) & (mod_nadp.time >= begDate) & (mod_nadp.time < endDate), 'SamplingInt'] = j 

mod_nadp.drop_duplicates(inplace = True)
mod_nadp.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.TimeseriesTMQ.SamplingInt.csv')
mod_nadp.head(10)

 59%|██████████████████████████████▊                     | 201/339 [11:00:25<4:26:57, 116.07s/sites]